# SilverLoader

Reusable incremental loader for Silver-layer tables. Provides `incremental_silver_load()` which performs a full load on first run and watermark-based incremental MERGE on subsequent runs. Uses `_batch_watermark` to track processed bronze rows (unaffected by dedup).

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


def incremental_silver_load(
    bronze_table_name,
    silver_table_name,
    key_columns,
    transform_fn,
    watermark_col="_ingestion_timestamp",
    update_condition=None,
):
    """Incrementally load data from bronze to silver layer.

    First run  -> full load (creates the silver table).
    Later runs -> reads only new bronze rows (watermark-based) and MERGE-upserts.
    Uses _batch_watermark to track the max ingestion timestamp of ALL rows
    processed in each batch (not affected by dedup dropping rows).

    Parameters
    ----------
    bronze_table_name : str   — fully qualified bronze table name.
    silver_table_name : str   — fully qualified silver table name.
    key_columns : list[str]   — column(s) used as the MERGE join key.
    transform_fn : callable  — (bronze_df) -> silver_df; must carry
                                _bronze_ingestion_ts through to its output.
    watermark_col : str       — bronze column for incremental detection
                                (default _ingestion_timestamp).
    update_condition : str|None — optional SQL condition for whenMatchedUpdate
                                so older source rows don't overwrite newer ones.

    Returns
    -------
    dict  — {"mode": "full"|"incremental", "rows_read": int, ...}
    """

    def _build_silver_df(bronze_df):
        """Add metadata columns and compute batch watermark before transform."""
        bronze_df = bronze_df.withColumn("_bronze_ingestion_ts", F.col(watermark_col))
        # Compute max watermark across ALL rows (before dedup drops any)
        batch_watermark = bronze_df.agg(F.max(watermark_col).alias("mw")).collect()[0]["mw"]
        silver_df = transform_fn(bronze_df)
        silver_df = (
            silver_df
            .withColumn("_silver_loaded_at", F.current_timestamp())
            .withColumn("_batch_watermark", F.lit(batch_watermark))
        )
        return silver_df, batch_watermark

    silver_exists = spark.catalog.tableExists(silver_table_name)

    # ------------------------------------------------------------------ #
    #  FULL LOAD  (silver table does not exist yet)                        #
    # ------------------------------------------------------------------ #
    if not silver_exists:
        bronze_df = spark.read.table(bronze_table_name)
        silver_df, batch_watermark = _build_silver_df(bronze_df)

        row_count = silver_df.count()
        (
            silver_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table_name)
        )

        print(f"[SilverLoader] FULL LOAD -> {row_count} rows written to {silver_table_name} (watermark: {batch_watermark})")
        return {"mode": "full", "rows_read": row_count, "rows_written": row_count, "watermark": str(batch_watermark)}

    # ------------------------------------------------------------------ #
    #  SCHEMA CHECK  (migrate old tables that lack _batch_watermark)       #
    # ------------------------------------------------------------------ #
    silver_cols = [f.name for f in spark.read.table(silver_table_name).schema.fields]

    if "_batch_watermark" not in silver_cols:
        print("[SilverLoader] _batch_watermark column missing; performing full refresh with schema update")
        bronze_df = spark.read.table(bronze_table_name)
        silver_df, batch_watermark = _build_silver_df(bronze_df)

        row_count = silver_df.count()
        (
            silver_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(silver_table_name)
        )
        return {"mode": "full_refresh", "rows_read": row_count, "rows_written": row_count, "watermark": str(batch_watermark)}

    # ------------------------------------------------------------------ #
    #  INCREMENTAL LOAD  (merge only new / changed rows)                   #
    # ------------------------------------------------------------------ #
    max_ts = (
        spark.read.table(silver_table_name)
        .agg(F.max("_batch_watermark").alias("max_ts"))
        .collect()[0]["max_ts"]
    )

    if max_ts is None:
        print("[SilverLoader] Watermark is null; performing full refresh")
        bronze_df = spark.read.table(bronze_table_name)
        silver_df, batch_watermark = _build_silver_df(bronze_df)

        row_count = silver_df.count()
        (
            silver_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(silver_table_name)
        )
        return {"mode": "full_refresh", "rows_read": row_count, "rows_written": row_count, "watermark": str(batch_watermark)}

    # Read only new bronze rows
    bronze_df = (
        spark.read.table(bronze_table_name)
        .filter(F.col(watermark_col) > F.lit(max_ts))
    )
    new_count = bronze_df.count()

    if new_count == 0:
        print(f"[SilverLoader] INCREMENTAL -> 0 new rows (watermark: {max_ts}). Skipping.")
        return {"mode": "incremental", "rows_read": 0, "rows_upserted": 0, "rows_inserted": 0, "watermark": str(max_ts)}

    silver_df, batch_watermark = _build_silver_df(bronze_df)

    merge_condition = " AND ".join(
        f"target.{k} = source.{k}" for k in key_columns
    )

    delta_table = DeltaTable.forName(spark, silver_table_name)

    merge_builder = (
        delta_table.alias("target")
        .merge(silver_df.alias("source"), merge_condition)
    )

    if update_condition:
        (
            merge_builder
            .whenMatchedUpdateAll(condition=update_condition)
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        (
            merge_builder
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    print(f"[SilverLoader] INCREMENTAL -> {new_count} rows merged into {silver_table_name} (old watermark: {max_ts}, new watermark: {batch_watermark})")
    return {"mode": "incremental", "rows_read": new_count, "rows_upserted": new_count, "watermark": str(batch_watermark)}